<a href="https://colab.research.google.com/github/saopa28/INSE-6450/blob/main/Milestone%204/Run_Full_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 4, deliverable 1: run the full pipeline in Colab
This notebook has no pipeline code of its own. It runs `run_pipeline.py` (which uses the `uplift_pipeline/` package) for every stage: model, robustness, monitoring, continual learning and active learning. Then it displays the saved results from `outputs/`. About 60 minutes on Colab CPU; add `--frac 0.2` to the command for a quick check.

In [1]:
# SETUP: install the XGBoost version the saved model was written with, and make the pipeline importable.
# The package is looked for (1) next to this notebook, (2) in an uploaded Milestone_4_code.zip,
# (3) in a clone of the GitHub repository (pulled again if it already exists).
%pip install -q xgboost==3.4.1 psutil
import os, sys, glob, zipfile, subprocess
PKG = None
if os.path.isdir("uplift_pipeline"):
    PKG = "."
if PKG is None and glob.glob("Milestone_4_code*.zip"):
    zipfile.ZipFile(sorted(glob.glob("Milestone_4_code*.zip"))[-1]).extractall(".")
    PKG = "Milestone 4"
if PKG is None:
    if os.path.isdir("INSE-6450"):
        subprocess.run(["git", "-C", "INSE-6450", "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/saopa28/INSE-6450.git"], check=True)
    PKG = os.path.join("INSE-6450", "Milestone 4")
if not os.path.isdir(os.path.join(PKG, "uplift_pipeline")):
    raise FileNotFoundError(f"uplift_pipeline not found in {PKG!r}. Push the 'Milestone 4' folder to GitHub, "
                            "or upload Milestone_4_code.zip to the Colab file panel and run this cell again.")
sys.path.insert(0, os.path.abspath(PKG))
import xgboost; print("xgboost", xgboost.__version__, "| pipeline from", os.path.abspath(PKG))

xgboost 3.4.1 | pipeline from /content/INSE-6450/Milestone 4


Add command --frac 0.2 at the end of the cell below to run only 20% of the rows for a fast check.

In [2]:
!python "{PKG}/run_pipeline.py" --out outputs

[data] loading https://github.com/saopa28/INSE-6450/releases/download/v1.0-engineered/criteo_uplift_engineered.zip (frac=1.0)
[data] train 9,785,714 | val 2,096,939 | test 2,096,939 rows
[model] v1 test {'qini': 0.0003758471418635433, 'auuc': 0.0004380917394296762, 'uplift@10%': 0.007953517606105846}
                     qini      auuc  uplift@10% preflight  qini_%_of_clean
scenario                                                                  
Clean            0.000376  0.000438    0.007954        OK            100.0
Noise 0.5 sd     0.000341  0.000399    0.007425     BLOCK             90.8
Rounded values   0.000277  0.000323    0.006985     BLOCK             73.8
Masking 10%      0.000289  0.000338    0.006351     BLOCK             76.8
Feature loss f9  0.000381  0.000444    0.007734     BLOCK            101.5
OOD shift +3 sd -0.000041 -0.000048    0.004369     BLOCK            -10.8
           batch status    psi_max psi_feature  oor_rate                                          

In [3]:
import pandas as pd, json
print(json.load(open('outputs/pipeline_summary.json')))
for f in ['robustness.csv', 'monitoring.csv', 'continual/continual_steps.csv', 'active_learning.csv', 'label_curation.csv']:
    print('\n' + f); display(pd.read_csv('outputs/' + f).round(6))

{'v1_test': {'qini': 0.0003758471418635433, 'auuc': 0.0004380917394296762, 'uplift@10%': 0.007953517606105846}, 'v1_trees': {'mu1': 244, 'mu0': 55, 'tau1': 84, 'tau0': 19}, 'rows': 13979592, 'label_curation': {'fakes': 3397, 'real_conversions': 8493, 'bot_cluster': 32, 'clean_qini': 0.00037947905568949425, 'noisy_qini': 0.00033957386392755056}, 'runtime_s': 2522.2}

robustness.csv


,scenario,qini,auuc,uplift@10%,preflight,qini_%_of_clean
0,Clean,0.000376,0.000438,0.007954,OK,100.0
1,Noise 0.5 sd,0.000341,0.000399,0.007425,BLOCK,90.8
2,Rounded values,0.000277,0.000323,0.006985,BLOCK,73.8
3,Masking 10%,0.000289,0.000338,0.006351,BLOCK,76.8
4,Feature loss f9,0.000381,0.000444,0.007734,BLOCK,101.5
5,OOD shift +3 sd,-0.000041,-0.000048,0.004369,BLOCK,-10.8



monitoring.csv


,batch,status,psi_max,psi_feature,oor_rate,alerts,qini,auuc,uplift@10%
0,Clean,OK,0.000030,f0,0.000025,NaN,0.000376,0.000438,0.007954
1,f9 +0.5 sd,BLOCK,13.282836,f9_x_f11,0.000039,BLOCK: PSI >= 0.25 -> run Qini check,0.000376,0.000438,0.008040
2,All raw +1 sd,BLOCK,13.305718,f9_x_f11,0.998045,BLOCK: out-of-range rows; BLOCK: PSI >= 0.25 -...,0.000377,0.000439,0.008078
3,All raw +3 sd,BLOCK,14.078094,f4_x_f9,1.000000,BLOCK: out-of-range rows; BLOCK: PSI >= 0.25 -...,-0.000097,-0.000115,0.003680



continual/continual_steps.csv


,step,drift,strategy,version_before,psi_max,psi_feature,monitor,window_qini_1,window_qini_2,triggered,...,qini_original,uplift10_original,auuc_original,update_s,peak_rss_mb,rss_delta_mb,gate_ci_lo,gate_ci_hi,gate_p_better,cand_trees
0,0,Clean,static,v1,0.000039,f8,OK,0.000410,0.000610,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,Clean,retrain_new,v1,0.000039,f8,OK,0.000410,0.000610,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,Clean,retrain_replay,v1,0.000039,f8,OK,0.000410,0.000610,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,Clean,warm_replay,v1,0.000039,f8,OK,0.000410,0.000610,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,f9 +0.5 sd,static,v1,13.284343,f9_x_f11,BLOCK,0.000377,0.000390,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,f9 +0.5 sd,retrain_new,v1,13.284343,f9_x_f11,BLOCK,0.000377,0.000390,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,f9 +0.5 sd,retrain_replay,v1,13.284343,f9_x_f11,BLOCK,0.000377,0.000390,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,f9 +0.5 sd,warm_replay,v1,13.284343,f9_x_f11,BLOCK,0.000377,0.000390,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2,All raw +1 sd,static,v1,13.305545,f9_x_f11,BLOCK,0.000384,0.000387,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2,All raw +1 sd,retrain_new,v1,13.305545,f9_x_f11,BLOCK,0.000384,0.000387,False,...,0.000376,0.007954,0.000438,NaN,NaN,NaN,NaN,NaN,NaN,NaN



active_learning.csv


,strategy,cycle,labeled_users,candidate_qini,candidate_uplift10,promoted,gate_ci_lo,gate_ci_hi,deployed_qini,deployed_uplift10,scoring_trees,scoring_us_per_user_batch,update_s
0,random,0,489285,0.000290,0.007154,NaN,NaN,NaN,0.00029,0.007154,38,0.792,0.0
1,random,1,978570,0.000339,0.007466,False,-0.000296,0.000175,0.00029,0.007154,38,0.892,51.2
2,random,2,1467855,0.000310,0.007274,False,-0.000067,0.000391,0.00029,0.007154,38,0.778,47.6
3,random,3,1957140,0.000372,0.007698,False,-0.000284,0.000138,0.00029,0.007154,38,0.792,48.9
4,random,4,2446425,0.000336,0.007618,False,-0.000232,0.000149,0.00029,0.007154,38,0.780,47.5
5,random,5,2935710,0.000319,0.007326,False,-0.000144,0.000290,0.00029,0.007154,38,0.923,48.7
6,active (hybrid),0,489285,0.000290,0.007154,NaN,NaN,NaN,0.00029,0.007154,38,0.785,0.0
7,active (hybrid),1,978570,0.000383,0.008008,False,-0.000199,0.000163,0.00029,0.007154,38,0.784,50.0
8,active (hybrid),2,1467855,0.000385,0.007877,False,-0.000263,0.000164,0.00029,0.007154,38,0.789,84.1
9,active (hybrid),3,1957140,0.000377,0.008263,False,-0.000274,0.000139,0.00029,0.007154,38,0.788,75.9



label_curation.csv


,strategy,cycle,reviewed_conversions,fakes_left,candidate_qini,candidate_pct_of_clean,promoted,gate_ci_lo,gate_ci_hi,deployed_qini,deployed_pct_of_clean,deployed_uplift10,update_s
0,random review,0,0,3397,0.000340,89.484218,NaN,NaN,NaN,0.00034,89.484218,0.006952,0.0
1,random review,1,1783,2883,0.000368,97.071600,False,-0.000101,0.000063,0.00034,89.484218,0.006952,107.3
2,random review,2,3566,2346,0.000373,98.195077,False,-0.000142,0.000089,0.00034,89.484218,0.006952,128.5
3,random review,3,5349,1848,0.000389,102.633037,False,-0.000148,0.000091,0.00034,89.484218,0.006952,127.1
4,targeted review,0,0,3397,0.000340,89.484218,NaN,NaN,NaN,0.00034,89.484218,0.006952,0.0
5,targeted review,1,1783,1724,0.000381,100.274543,False,-0.000155,0.000092,0.00034,89.484218,0.006952,127.9
6,targeted review,2,3566,180,0.000391,103.013860,False,-0.000244,0.000047,0.00034,89.484218,0.006952,120.1
7,targeted review,3,5349,180,0.000364,95.983169,False,-0.000152,0.000089,0.00034,89.484218,0.006952,138.4
